## Importing packages

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, TimeSeriesSplit
from sklearn.feature_selection import SelectKBest, f_regression, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    mean_absolute_error,
)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.compose import TransformedTargetRegressor

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN, SMOTETomek

from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.model_selection import KFold

import optuna
from optuna.samplers import TPESampler

import matplotlib.pyplot as plt
import joblib
from datetime import datetime

import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("../data_model/db_additional_fe.csv")
df_test = pd.read_csv("../data_model/db_collected_fe.csv")

def sanitize_column_names(df):
    """Replace special JSON characters in column names with underscores."""
    import re
    sanitized = {}
    for col in df.columns:
        sanitized_col = re.sub(r'[\[\](){},\"\'\\\/\s]', '_', str(col))
        sanitized_col = re.sub(r'_+', '_', sanitized_col)
        sanitized_col = sanitized_col.strip('_')
        sanitized[col] = sanitized_col
    df = df.rename(columns=sanitized)
    return df

df = sanitize_column_names(df)
df_test = sanitize_column_names(df_test)

print(f"Data loaded: Train {df.shape}, Test {df_test.shape}")

In [ ]:
y_train_clf = (df['volume_total'] > 10_000).astype(int)
X_train_clf = df.drop(columns=['volume_total'])

y_test_clf = (df_test['volume_total'] > 10_000).astype(int)
X_test_clf = df_test.drop(columns=['volume_total'])

print(f"Classification data - Train: {X_train_clf.shape}, Test: {X_test_clf.shape}")
print(f"Train distribution: {(y_train_clf==0).sum()} negative ({(y_train_clf==0).sum()/len(y_train_clf):.1%}), "
      f"{(y_train_clf==1).sum()} positive ({(y_train_clf==1).sum()/len(y_train_clf):.1%})")
print(f"Test distribution: {(y_test_clf==0).sum()} negative ({(y_test_clf==0).sum()/len(y_test_clf):.1%}), "
      f"{(y_test_clf==1).sum()} positive ({(y_test_clf==1).sum()/len(y_test_clf):.1%})")

X_train_clf = X_train_clf.fillna(0)
X_test_clf = X_test_clf.fillna(0)

In [ ]:
pos_count = (y_train_clf == 1).sum()
neg_count = (y_train_clf == 0).sum()
total_count = len(y_train_clf)

current_pos_ratio = pos_count / total_count
current_neg_ratio = neg_count / total_count

target_pos_ratio = 0.05 #from test set
target_neg_ratio = 0.95 #from test set

weight_pos = target_pos_ratio / current_pos_ratio
weight_neg = target_neg_ratio / current_neg_ratio

sample_weights = np.where(y_train_clf == 1, weight_pos, weight_neg)

print(f"Train distribution: {neg_count} negative ({current_neg_ratio:.1%}), {pos_count} positive ({current_pos_ratio:.1%})")
print(f"Target distribution: 95% negative, 5% positive")
print(f"Weight for class 0 (negative): {weight_neg:.3f}")
print(f"Weight for class 1 (positive): {weight_pos:.3f}")
print(f"Sample weights shape: {sample_weights.shape}")
print(f"Sample weights ready")

In [ ]:
models_sampling = {
    "xgb_weighted": {
        "samplers": [],
        "classifier": XGBClassifier(
            objective="binary:logistic",
            tree_method="hist",
            eval_metric="aucpr",
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
        ),
    },
    
    "rf_weighted": {
        "samplers": [],
        "classifier": RandomForestClassifier(
            n_jobs=N_JOBS,
            random_state=RANDOM_STATE,
        ),
    },
}

classifier_constructors = {
    "xgb": lambda: XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",
        eval_metric="aucpr",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
    ),
    
    "rf": lambda: RandomForestClassifier(
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
    ),
}

param_spaces_clf = {
    "xgb": {
        "n_estimators": ("int", 100, 500),
        "max_depth": ("int", 4, 8),
        "learning_rate": ("float", 0.01, 0.1),
        "subsample": ("float", 0.7, 1.0),
        "colsample_bytree": ("float", 0.7, 1.0),
        "reg_alpha": ("float", 0.1, 1.0),
        "reg_lambda": ("float", 0.5, 5.0),
    },
    
    "rf": {
        "n_estimators": ("int", 100, 500),
        "max_depth": ("int", 5, 15),
        "min_samples_split": ("int", 5, 20),
        "min_samples_leaf": ("int", 2, 10),
    },
}

print("Classification models configured")

In [ ]:
def create_classification_objective(name, samplers, X_train, y_train, cv):
    """Create Optuna objective - fixed for sklearn 1.8+"""
    
    model_type = get_model_type(name)
    
    def objective(trial):
        params = {}
        for param_name, param_config in param_spaces_clf[model_type].items():
            param_type = param_config[0]
            
            if param_type == "int":
                params[param_name] = trial.suggest_int(param_name, param_config[1], param_config[2])
            elif param_type == "float":
                params[param_name] = trial.suggest_float(param_name, param_config[1], param_config[2])
            elif param_type == "categorical":
                params[param_name] = trial.suggest_categorical(param_name, param_config[1])
        
        clf = classifier_constructors[model_type]()
        clf.set_params(**params)
        
        scores = cross_val_score(
            clf, X_train, y_train,
            cv=cv,
            scoring="roc_auc",
            n_jobs=N_JOBS
        )
        
        return scores.mean()
    
    return objective

print("Fixed objective function ready")

In [ ]:
cv_clf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results_sampling = {}

for name, config in models_sampling.items():
    print(f"\n{'='*60}")
    print(f"Training: {name.upper()}")
    print(f"{'='*60}")
    
    model_type = get_model_type(name)
    
    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=RANDOM_STATE),
        study_name=f"{name}_classification"
    )
    
    objective = create_classification_objective(name, [], X_train_clf, y_train_clf, cv_clf)
    
    print(f"Running {N_TRIALS} trials...")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    
    if len(study.trials) == 0 or study.best_trial is None:
        print(f"WARNING: No trials completed for {name}, skipping...")
        continue
    
    best_params = study.best_params
    
    clf = classifier_constructors[model_type]()
    clf.set_params(**best_params)
    clf.fit(X_train_clf, y_train_clf, sample_weight=sample_weights)
    
    y_pred_proba = clf.predict_proba(X_test_clf)[:, 1]
    
    precisions, recalls, thresholds = precision_recall_curve(y_test_clf, y_pred_proba)
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)
    best_threshold_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_threshold_idx] if len(thresholds) > best_threshold_idx else 0.5
    
    y_pred = (y_pred_proba >= best_threshold).astype(int)
    
    results_sampling[name] = {
        "study": study,
        "model": clf,
        "best_params": best_params,
        "best_threshold": best_threshold,
        "cv_roc_auc": study.best_value,
        "test_recall": recall_score(y_test_clf, y_pred),
        "test_precision": precision_score(y_test_clf, y_pred, zero_division=0),
        "test_f1": f1_score(y_test_clf, y_pred),
        "test_pr_auc": average_precision_score(y_test_clf, y_pred_proba),
    }
    
    print(f"\nOptimization complete!")
    print(f"  Best CV ROC-AUC: {study.best_value:.4f}")
    print(f"  Test PR-AUC: {results_sampling[name]['test_pr_auc']:.4f}")
    print(f"  Test F1: {results_sampling[name]['test_f1']:.4f}")

In [ ]:
print(f"\n{'='*80}")
print(f"{'='*20} CLASSIFICATION SUMMARY {'='*20}")
print(f"{'='*80}\n")

print(f"{'Model':<20} {'CV PR-AUC':<12} {'Test PR-AUC':<12} {'Recall':<10} {'Precision':<12} {'F1':<10} {'Threshold':<10}")
print("-" * 110)

for name, result in results_sampling.items():
    cv_pr = result['cv_pr_auc']
    test_pr = result['test_pr_auc']
    recall = result['test_recall']
    precision = result['test_precision']
    f1 = result['test_f1']
    threshold = result['best_threshold']
    
    print(f"{name:<20} {cv_pr:<12.4f} {test_pr:<12.4f} {recall:<10.4f} {precision:<12.4f} {f1:<10.4f} {threshold:<10.3f}")

best_pr_auc_name = max(results_sampling.items(), key=lambda x: x[1]['test_pr_auc'])[0]
print(f"\n Best model by PR-AUC: {best_pr_auc_name.upper()} (PR-AUC: {results_sampling[best_pr_auc_name]['test_pr_auc']:.4f})")

best_f1_name = max(results_sampling.items(), key=lambda x: x[1]['test_f1'])[0]
print(f" Best model by F1: {best_f1_name.upper()} (F1: {results_sampling[best_f1_name]['test_f1']:.4f})")

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save best regression model
best_reg_name = min(results_regression.items(), key=lambda x: x[1]['test_rmse'])[0]
best_reg_model = results_regression[best_reg_name]["model"]

reg_filename = f"best_regressor_{best_reg_name}_{timestamp}.pkl"
joblib.dump(best_reg_model, reg_filename)
print(f"✅ Best regression model saved: {reg_filename}")

# Save best classification model
best_clf_name = max(results_sampling.items(), key=lambda x: x[1]['test_pr_auc'])[0]
best_clf_pipe = results_sampling[best_clf_name]["best_pipe"]

clf_filename = f"best_classifier_{best_clf_name}_{timestamp}.pkl"
joblib.dump(best_clf_pipe, clf_filename)
print(f"✅ Best classifier saved: {clf_filename}")

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE!")
print(f"{'='*60}")